# 技能3 · Day 1 上机：在真实数据上识别混杂、估计因果效应

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 在**真实数据**（Lalonde/NSW）上区分"朴素均值差（有偏）"与"后门调整估计（因果）"
2. 用 DoWhy 完成"建模→识别→估计→反驳"四步因果分析
3. 解释混杂偏差的来源，并用反驳检验验证估计稳健性

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实数据集：Lalonde/NSW（NSW 职业培训实验真实数据），营销映射见下。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml -q

## 1. 数据集背景与营销映射

**Lalonde/NSW 数据集**：NSW 职业培训示范实验的真实数据（Dehejia & Wahba 1999），因果推断最经典的真实教学数据集，**真实存在严重混杂**。

| NSW 变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到优惠券/看到广告 | 处理 T |
| `re78` | 转化率 / GMV / 客单价 | 结果 Y |
| `age`,`education`,`re74`,`re75`,`black`,`hispanic`,`married`,`nodegree` | 用户画像 / 历史消费 | 协变量 X（潜在混杂）|

**因果问题**：参加培训（收到优惠券）对收入（转化）的**真实因果效应**是多少？朴素均值差为何有偏？

In [ ]:
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel
from causaldata import nsw
import warnings
warnings.filterwarnings('ignore')

## 1-2：加载与探索真实数据

In [ ]:
# 1. 加载真实 NSW 数据
df = nsw.load_pandas().data
print(f"数据形状: {df.shape}")
df.head()

In [ ]:
# 2. 探索数据 —— 处理组/对照组协变量均衡性
print("处理组样本量:", len(df[df['treat']==1]))
print("对照组样本量:", len(df[df['treat']==0]))
print()

# 关键协变量分组均值对比
covariates = ['age', 'education', 'black', 're74', 're75']
balance = df.groupby('treat')[covariates].mean().T
balance.columns = ['对照组(treat=0)', '处理组(treat=1)']
balance['差值'] = balance['处理组(treat=1)'] - balance['对照组(treat=0)']
print("协变量均衡性对比：")
print(balance)
print()
print("⚠️ 观察：处理组与对照组在 age/education/re75 上分布不均 → 存在混杂")

## 2. 因果图（DAG）与混杂分析

NSW 场景的因果结构（简化）：

```
age, education ──> treat ──> re78
       │              ▲
       └──────────────┘   (后门路径)
re74, re75 ──> treat ──> re78
       │            ▲
       └────────────┘
```

- **混杂因素**：`age`/`education`/`re74`/`re75` 等同时影响 `treat` 和 `re78`
- **后门路径**：treat ← {age, education, re74, re75, ...} → re78
- **后门准则**：控制这些协变量即可阻断后门路径，识别 treat→re78 的因果效应

**营销映射**：用户活跃度（对应 `re75` 历史消费）既影响"是否收到优惠券"(treat)，又影响"是否转化"(re78)——典型混杂。

## 3：朴素估计（有偏）

In [ ]:
# 3. 朴素估计（有偏）
naive_ate = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"朴素估计 ATE = {naive_ate:.2f}")
print("⚠️ 这个估计有偏！因为处理组与对照组协变量分布不均（见上）")

## 3. 为什么朴素估计有偏

朴素估计 = ATE + Bias。偏差来自混杂因素在两组分布不均。

例如：参加培训的人本来 `education` 更高、`re75`（前期收入）更高，那么处理组 `re78` 更高可能不是培训的功劳，而是他们条件本就更好。

**营销类比**：收到优惠券的用户可能本来就是高活跃用户（自选择），他们的高转化率可能不是优惠券的功劳，而是他们本来就会买。

→ 需要用**后门调整**（控制混杂）消除偏差。下面用 DoWhy。

## 4-5：DoWhy 因果分析

In [ ]:
# 4. DoWhy 建模 → 识别 → 估计（后门调整）
common_causes = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]

model = CausalModel(
    data=df,
    treatment="treat",
    outcome="re78",
    common_causes=common_causes
)

identified_estimand = model.identify_effect()
print("识别出的估计量：")
print(identified_estimand)
print()

causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression"
)
print(f"因果估计 ATE = {causal_estimate.value:.2f}")
print(f"（对比朴素估计 {naive_ate:.2f}）—— 差异来自混杂偏差")

In [ ]:
# 5. 反驳检验（安慰剂处理）
refutation = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    "placebo_treatment_refuter"
)
print(refutation)
print()
print("解读：安慰剂处理下，新估计应接近 0 —— 若如此，说明方法没在虚假处理上'发现'效应，估计可靠。")

## 4. 营销延伸：倾向得分匹配（PSM）

后门调整（线性回归）是一种方法。PSM 是另一种常用的观测数据因果估计法：按"收到处理的概率"（倾向得分）把处理组与对照组匹配，再算匹配后的均值差。

在营销中，PSM 常用于把"收到优惠券的用户"与"相似但没收到优惠券的用户"匹配，估计优惠券的真实增量效应。

In [ ]:
# 6（可选）：PSM 再估一次
psm_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching"
)
print(f"PSM 估计 ATE = {psm_estimate.value:.2f}")
print(f"三种估计对比：朴素 {naive_ate:.2f} | 后门回归 {causal_estimate.value:.2f} | PSM {psm_estimate.value:.2f}")
print()
print("解读：朴素估计与后门/PSM 估计的差异 = 混杂偏差。后门与 PSM 两种方法的接近程度 = 估计稳健性。")

## 5. 反思与前沿

### 反思问题
1. 朴素估计与后门调整估计的差异，主要来自哪个混杂变量？（提示：看 TODO2 的均衡性对比，哪 个协变量两组差距最大）
2. 安慰剂检验结果是否支持你的因果估计？（若安慰剂效应≈0，说明方法没在虚假处理上"发现"效应，方法可靠）
3. 如果 NSW 数据里有个**没观测到的混杂**（如"个人上进心"），你的估计还可靠吗？→ 这是"可忽略性"假设的根本局限

### 2026 前沿：LLM-as-a-judge 自检因果论证
把你建好的 DAG + 识别策略 + 估计 + 反驳结果整理成一段结构化描述，让 LLM 扮演"因果推断评审"，检查：
- DAG 是否遗漏了可能的混杂？
- 识别策略是否满足后门准则？
- 反驳检验是否充分？
- 结论是否过度外推？

参考 arXiv 2306.05685（NeurIPS 2023, LLM-as-a-judge）。**注意**：LLM 只审查论证质量，不估计效应本身——它停留在因果阶梯 L1，不能上升到 L2/L3。